# Lab 5 - Multiagent Pattern for Transmission and Custody Review

This is the main Lab 5 forensic notebook. If you want a gentler introduction first, start with [03a_program_promotion_demo.ipynb](./03a_program_promotion_demo.ipynb), which uses the same Multiagent Pattern in a non-forensics setting.

In this notebook, the three agent roles match the Lab 5 case more directly:
- `InvestigationAgent` reconstructs the technical timeline.
- `EvidenceVerificationAgent` checks whether the technical claims are actually supported by the artifacts.
- `CustodyAuditAgent` reviews the chain-of-custody record and writes the final evidence-bounded conclusion.


![Figure 1. Multiagent-pattern workflow for Lab 5](./figures/lab5_multiagent_workflow.svg)

This figure shows the Lab 5 workflow: the case package is split across specialized roles, the agent outputs are compared, and the final conclusion is only written after both the technical evidence and the evidence-handling record are reviewed.

## What You Will Do
1. Set up the notebook and point it to the Lab 5 artifact files.
2. Define simple case-specific tools for each artifact type.
3. Inspect the artifact list and a few example tool outputs.
4. Create three specialized agents with clear role boundaries.
5. Run the agents step by step so you can see how context moves across the team.
6. Let `CustodyAuditAgent` write the final confidence-bounded case conclusion.
7. Recreate the same dependency graph with `Crew` and run the full workflow again.


## Lab Question and Role Outputs

Use the Lab 5 artifacts to answer this case question:

**Was `patients_contacts.png` transmitted from the outreach phone, should the conclusion be labeled `confirmed`, `likely`, or `unconfirmed`, and does the missing transfer in the chain-of-custody log weaken confidence in that conclusion?**

Keep these role-specific outputs visible as you work:
- `InvestigationAgent` should return: `Investigation timeline`, `Initial transmission assessment`, `Open technical questions`.
- `EvidenceVerificationAgent` should return: `Claims checked`, `Supported evidence`, `Unsupported or missing evidence`, `Revised transmission rating`.
- `CustodyAuditAgent` should return: `Verified technical finding`, `Custody review`, `Confidence impact`, `Final conclusion`.

Important: downstream agents automatically receive the earlier agents' outputs as context. That context acts like short-term working notes inside the multiagent workflow.


### Step 1: Set Up the Notebook

Run this setup cell first. It loads the Lab 5 environment variables, checks that you opened the notebook from the correct folder, adds `src/` to the import path, and points the notebook to the artifact files in `data/`.


In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

LAB_NAME = 'lab5_multiagent_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Expected .env in this folder. Copy .env.example to .env first.')

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Expected a data/ folder in this lab directory.')

# Import the course-specific classes only after src/ has been added to the Python path.
from agentic_patterns.multiagent_pattern.agent import Agent
from agentic_patterns.multiagent_pattern.crew import Crew
from agentic_patterns.tool_pattern.tool import tool

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print('Data folder:', data_dir)
print('Model:', MODEL)


### Step 2: Define Small Evidence Tools

Each tool below reads one part of the Lab 5 case. The goal is to keep each tool simple and easy to explain:
- some tools return technical evidence about the device, file, messaging app, or network,
- one tool returns the chain-of-custody record,
- and one tool lists the available artifact files.

These tools are intentionally case-specific so students can focus on the Multiagent Pattern instead of learning a large generic tool library.


In [ ]:
def read_csv_rows(filename: str):
    # Read one CSV file into a list of small dictionaries so the notebook can show structured evidence.
    with (data_dir / filename).open(newline='') as handle:
        return list(csv.DictReader(handle))


@tool
def list_artifact_files():
    """Return the available artifact files for Lab 5."""
    return sorted(path.name for path in data_dir.iterdir() if path.is_file())


@tool
def get_device_state_events():
    """Return the lock and unlock records that bound the unattended interval."""
    return read_csv_rows('device_state.csv')


@tool
def get_patients_file_event():
    """Return the file event for patients_contacts.png."""
    return read_csv_rows('file_events.csv')


@tool
def get_securechat_attachment_event():
    """Return the SecureChat attachment-attempt record for patients_contacts.png."""
    return read_csv_rows('messaging_events.csv')


@tool
def get_securechat_network_event():
    """Return the SecureChat network upload record."""
    return read_csv_rows('network_events.csv')


@tool
def get_chain_of_custody_events():
    """Return the chain-of-custody records used to judge evidence-handling completeness."""
    return read_csv_rows('chain_of_custody.csv')


technical_tools = [
    list_artifact_files,
    get_device_state_events,
    get_patients_file_event,
    get_securechat_attachment_event,
    get_securechat_network_event,
]

custody_tools = [
    list_artifact_files,
    get_chain_of_custody_events,
]


### Step 3: Inspect the Artifact List and Example Tool Outputs

Before asking the agents to work, it helps to look at a few tool results directly. This makes it easier to understand what kind of evidence the agents will be reasoning over.


In [ ]:
artifact_files = list_artifact_files.run()
attachment_example = get_securechat_attachment_event.run()
custody_example = get_chain_of_custody_events.run()

display(Markdown('### Artifact Files\n\n```json\n' + json.dumps(artifact_files, indent=2) + '\n```'))
display(Markdown('### Example Messaging Record\n\n```json\n' + json.dumps(attachment_example, indent=2) + '\n```'))
display(Markdown('### Example Custody Record\n\n```json\n' + json.dumps(custody_example, indent=2) + '\n```'))


### Step 4: Create the Three Forensic Agents

This cell defines the three Lab 5 roles and connects their dependencies.

A few plain-language notes before you run it:
- `InvestigationAgent` focuses on the technical story first.
- `EvidenceVerificationAgent` checks whether the technical story is actually supported by the records.
- `CustodyAuditAgent` receives both earlier outputs, audits the handling record, and writes the final bounded conclusion.

The arrows mean that later agents automatically receive earlier outputs as context.


In [ ]:
LAB_QUESTION = (
    'Was patients_contacts.png transmitted from the outreach phone, should the conclusion be labeled '
    'confirmed, likely, or unconfirmed, and does the missing transfer in the chain-of-custody log weaken '
    'confidence in that conclusion?'
)

CASE_SCOPE = """
Case facts:
- Incident window: 2026-03-01T18:35:00Z to 2026-03-01T19:10:00Z.
- File of interest: patients_contacts.png.
- Technical artifacts include device-state, file, messaging, and network records.
- Evidence-handling review uses chain_of_custody.csv.
- Final conclusion must use one label: confirmed, likely, or unconfirmed.
""".strip()


def build_forensic_agents():
    investigation_agent = Agent(
        name='InvestigationAgent',
        backstory=(
            'You reconstruct mobile-forensics timelines. '
            'Focus on ordering technical events carefully and avoid overclaiming upload completion.'
        ),
        task_description=(
            f'{LAB_QUESTION}\n\n'
            f'{CASE_SCOPE}\n\n'
            'Use the technical evidence tools to reconstruct the timeline and produce an initial transmission assessment.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Investigation timeline:\n'
            'Initial transmission assessment:\n'
            'Open technical questions:'
        ),
        tools=technical_tools,
        llm=MODEL,
    )

    evidence_verification_agent = Agent(
        name='EvidenceVerificationAgent',
        backstory=(
            'You verify whether technical claims are supported by specific artifact records. '
            'You correct unsupported claims and keep the rating evidence-bounded.'
        ),
        task_description=(
            f'{LAB_QUESTION}\n\n'
            f'{CASE_SCOPE}\n\n'
            'Use the technical evidence tools to test the earlier technical claims. '
            'If the artifacts show only file creation, an attach attempt, and upload_started, say so clearly. '
            'Do not treat upload_started as confirmed completion.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Claims checked:\n'
            'Supported evidence:\n'
            'Unsupported or missing evidence:\n'
            'Revised transmission rating:'
        ),
        tools=technical_tools,
        llm=MODEL,
    )

    custody_audit_agent = Agent(
        name='CustodyAuditAgent',
        backstory=(
            'You review chain-of-custody records and explain how evidence-handling gaps affect final confidence. '
            'You use the earlier agents\' technical findings but focus on whether the handling record supports trust.'
        ),
        task_description=(
            f'{LAB_QUESTION}\n\n'
            f'{CASE_SCOPE}\n\n'
            'Use the chain-of-custody tool and the earlier agent context to write the final case conclusion. '
            'Explain how the missing transfer changes confidence, even if the technical evidence suggests likely transmission.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Verified technical finding:\n'
            'Custody review:\n'
            'Confidence impact:\n'
            'Final conclusion:'
        ),
        tools=custody_tools,
        llm=MODEL,
    )

    investigation_agent >> evidence_verification_agent
    investigation_agent >> custody_audit_agent
    evidence_verification_agent >> custody_audit_agent

    return investigation_agent, evidence_verification_agent, custody_audit_agent


def run_crew_with_markdown(crew: Crew):
    outputs = {}

    # Crew chooses a safe dependency order; we only improve the notebook display format.
    for agent in crew.topological_sort():
        output = agent.run()
        outputs[agent.name] = output
        display(Markdown(f'### {agent.name} Output\n\n{output}'))

    return outputs


investigation_agent, evidence_verification_agent, custody_audit_agent = build_forensic_agents()

print('InvestigationAgent dependents:', investigation_agent.dependents)
print('EvidenceVerificationAgent dependencies:', evidence_verification_agent.dependencies)
print('CustodyAuditAgent dependencies:', custody_audit_agent.dependencies)


### Step 5: Run `InvestigationAgent`

This first role reconstructs the technical sequence from the Lab 5 artifacts. Its output is preliminary on purpose: it should organize the timeline, but it should not treat `upload_started` as proof that the transfer finished.


In [ ]:
investigation_output = investigation_agent.run()
display(Markdown('### InvestigationAgent Output\n\n' + investigation_output))


### Step 6: Inspect and Run `EvidenceVerificationAgent`

The next cell first shows the context automatically passed from `InvestigationAgent`. Then it runs `EvidenceVerificationAgent`, whose job is to test the earlier claims against the technical records and correct anything that is too strong.


In [ ]:
display(Markdown('### EvidenceVerificationAgent Received Context\n\n```text\n' + evidence_verification_agent.context + '\n```'))

evidence_verification_output = evidence_verification_agent.run()
display(Markdown('### EvidenceVerificationAgent Output\n\n' + evidence_verification_output))


### Step 7: Inspect and Run `CustodyAuditAgent`

`CustodyAuditAgent` now receives both earlier technical outputs as context. It adds one more kind of analysis: whether the evidence-handling record is complete enough to trust the final conclusion.


In [ ]:
display(Markdown('### CustodyAuditAgent Received Context\n\n```text\n' + custody_audit_agent.context + '\n```'))

custody_audit_output = custody_audit_agent.run()
display(Markdown('### CustodyAuditAgent Final Conclusion\n\n' + custody_audit_output))


## Part 2: Recreate the Same Workflow with `Crew`

The manual run above makes each role easier to inspect. `Crew` packages the same idea more neatly by holding the dependency graph in one place.

The next cells create a fresh forensic team inside a `Crew` context, show the dependency graph, and run the same workflow again using `Crew`'s topological order.


In [ ]:
with Crew() as crew:
    crew_investigation_agent, crew_evidence_verification_agent, crew_custody_audit_agent = build_forensic_agents()


### Step 8: Visualize the Forensic Crew Graph

This graph shows the role flow for Lab 5:
- `InvestigationAgent` goes first,
- `EvidenceVerificationAgent` checks the technical story,
- `CustodyAuditAgent` runs last because it needs both earlier outputs before it writes the final confidence-bounded conclusion.


In [ ]:
crew.plot()


### Step 9: Run the Full Forensic Crew

This helper uses the dependency order from `Crew`, but displays each role output as Markdown so the notebook stays easier to read.


In [ ]:
crew_outputs = run_crew_with_markdown(crew)


## What To Notice

As you review the outputs, look for these points:
- Did `InvestigationAgent` keep the technical timeline clear without overstating completion?
- Did `EvidenceVerificationAgent` lower confidence when it saw that the artifacts only show `upload_started`, not `upload_completed`?
- Did `CustodyAuditAgent` explain why a missing transfer entry weakens confidence in the final conclusion?

Those three questions capture the main Lab 5 idea: multiagent collaboration is useful when different roles contribute different kinds of evidence checking before a final answer is written.
